In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Open by name (or use open_by_url / open_by_key if you have the sheet's URL/ID)
sheet = gc.open('Capstone Copy of Appointments').worksheet('Appointments_2025/2026')
tenure_sheet= gc.open('Capstone Copy of Appointments').worksheet('Clinician_Tenure')

# Pull all data into a DataFrame
import pandas as pd
import json
data = sheet.get_all_records()
raw_df = pd.DataFrame(data)

tenure = tenure_sheet.get_all_records()
tenure_df = pd.DataFrame(tenure)


In [ ]:
# Mount Drive first — everything below depends on this
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
Github_Token = userdata.get('Github_Token')
!git remote set-url origin https://{Github_Token}@github.com/Sharion2023/Data_Analytics_Capstone.git

# Move into the repo (already cloned, lives permanently in Drive)
%cd /content/drive/MyDrive/Data_Analytics_Capstone

# Git identity (session-only, still needs to be set each time)
!git config user.name "Sharion2023"
!git config user.email "your-email@example.com"

# Activate nbstripout locally each session
!pip install nbstripout --quiet
!nbstripout --install

# Load staff_anon.json from Drive
import json
with open('/content/drive/MyDrive/staff_anon.json', 'r') as f:
    staff_anon = json.load(f)

print("✅ Drive mounted, repo location set, git configured, nbstripout active, staff_anon loaded.")

In [ ]:
#create df for calculated results
si_ratio_sheet = gc.open('Capstone Copy of Appointments').worksheet('DataStudioSource_Practitioner')
si_data = si_ratio_sheet.get_all_records()
si_df = pd.DataFrame(si_data)

In [ ]:
# Strip whitespace on the join key in both DataFrames
raw_df['staff_member_name'] = raw_df['staff_member_name'].str.strip()
tenure_df['staff_member_name'] = tenure_df['staff_member_name'].str.strip()
si_df['staff_member_name'] = si_df['staff_member_name'].str.strip()

# Merge
df = raw_df.merge(tenure_df, on='staff_member_name', how='left')

print(f"raw_df rows: {len(raw_df)} | merged df rows: {len(df)}")

In [ ]:
unmatched_count = df[df['start_date'].isna()]['staff_member_name'].nunique()
print(f"{unmatched_count} unique staff members did not match")

In [ ]:
df = df.merge(si_df, on=['staff_member_name', 'iso_week'], how='left')

In [ ]:
#map to aonymous mapping
df['clinician_id'] = df['staff_member_name'].map(staff_anon)

In [ ]:
#check that all mapping worked
unmatched = df['clinician_id'].isna().sum()
print(f"{unmatched} rows failed to map to a clinician_id")

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:

df

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean.columns

In [ ]:
#drop unnecessary columns
col_to_keep = [
    'patient_number',    # this is your patient ID field, not 'patient_id'
    'clinician_id',
    'start_at',
    'arrived_at',
    'first_visit',
    'treatment_name',
    'booked_at', # lead time analysis for intake-conversion stream
    'start_date',
    'iso_week',
    'subsequent_visits',
    'initial_visits',
    'real_date',
       'weekly_S/I_ratio',
    'rolling_4-week_S/I_ratio',
    'tenure_status',
       'unique_patients',
    'fall_off_patients',
    '4_wk_fall_off_patient_calc'
]

df_clean = df_clean[col_to_keep]

In [ ]:
df_clean.head()
df_clean.isna().sum()


In [ ]:
df_clean['fall_off_patients'].unique()

In [ ]:
df_clean['start_date'] = pd.to_datetime(df_clean['start_date'], errors='coerce')
df_clean['start_at'] = pd.to_datetime(df_clean['start_at'], errors='coerce')
df_clean['arrived_at'] = pd.to_datetime(df_clean['arrived_at'],errors='coerce')
df_clean['booked_at'] = pd.to_datetime(df_clean['booked_at'], errors='coerce')
df_clean['real_date'] = pd.to_datetime(df_clean['real_date'], errors='coerce')
df_clean['weekly_S/I_ratio'] = pd.to_numeric(df_clean['weekly_S/I_ratio'], errors='coerce')
df_clean['rolling_4-week_S/I_ratio'] = pd.to_numeric(df_clean['rolling_4-week_S/I_ratio'], errors='coerce')
df_clean['fall_off_patients'] = pd.to_numeric(df_clean['fall_off_patients'], errors='coerce')
df_clean['4_wk_fall_off_patient_calc'] = pd.to_numeric(df_clean['4_wk_fall_off_patient_calc'], errors='coerce')

In [ ]:
df_clean.dtypes

In [ ]:
df_clean.sort_values('real_date', ascending=True).tail()

In [ ]:
df_clean.head(10)

In [ ]:
df_clean.shape

In [ ]:
df_clean['clinician_id'].unique()

**Let's calculate tenure for each clinician, and add column to df_clean for later use**

In [ ]:
#Calculate each practitioner's tenure with the clinic

reference_date = df_clean['real_date'].max()
df_clean['tenure_years'] = (reference_date - df_clean['start_date']).dt.days / 365.25

#Build summary table of all clinicians
tenure_si_summary = df_clean.groupby('clinician_id').agg(
    tenure_years=('tenure_years', 'first'),
    avg_si_ratio=('rolling_4-week_S/I_ratio', 'mean'),
    patients_per_clinician =('patient_number', 'nunique')
).reset_index()



In [ ]:
# Confirm total unique patients, expect 4309
print(df_clean['patient_number'].nunique())

In [ ]:
#Display total unique patients per clinician
print (tenure_si_summary)

In [ ]:
print(tenure_si_summary['patients_per_clinician'].sum())

In [ ]:
# Count how many times each patient appears across distinct practitioners
patients_multi = df_clean.groupby('patient_number')['clinician_id'].nunique()
print((patients_multi > 1).sum())  # number of patients seen by >1 practitioner
print(patients_multi[patients_multi > 1].sum() - (patients_multi > 1).sum())  # "extra" counts this creates

**Let's see patients who returned for at least a second visit**

In [ ]:
visits_per_patient_per_clinician = df_clean.groupby(['clinician_id', 'patient_number']).size().reset_index(name='visits')

#What is the number of patients who return for at least a second visit?
visits_per_patient_per_clinician['returned'] = visits_per_patient_per_clinician['visits'] > 1
print(visits_per_patient_per_clinician)
print(visits_per_patient_per_clinician['returned'].value_counts())

**What does the rate of patients who returned for at least a second visit look like on the clinic level?**

In [ ]:
return_rate_for_clinic = visits_per_patient_per_clinician['returned'].mean()
print(return_rate_for_clinic)


**What does the rate of patients who returned for at least a second visit look like on the clinician level?**

In [ ]:
return_rate_by_clinician = visits_per_patient_per_clinician.groupby('clinician_id')['returned'].mean().reset_index()
return_rate_by_clinician.columns = ['clinician_id', 'second_visit_return_rate']
print(return_rate_by_clinician)

In [ ]:
clinic_wide_sheet = gc.open('Capstone Copy of Appointments').worksheet('DataStudioSource_Clinic')
clinic_wide_data = clinic_wide_sheet.get_all_records()
clinic_wide_df = pd.DataFrame(clinic_wide_data)

**What does 4 week average S/I ratio look like for the clinic overall?**

In [ ]:
import matplotlib.pyplot as plt

#Calculate clinic wide S/I ratio

clinic_wide_df['rolling_4-week_S/I_ratio'] = pd.to_numeric(
    clinic_wide_df['rolling_4-week_S/I_ratio'], errors='coerce'
)

clinic_trend = clinic_wide_df.set_index('iso_week')['rolling_4-week_S/I_ratio']

fig, ax = plt.subplots(figsize=(10, 5))

clinic_trend.plot(ax=ax, marker='o', color='#4C72B0')
plt.title('Clinic 4 Week S/I Ratio Trend')
plt.xlabel('Week')
plt.ylabel('4 Week S/I Ratio')
plt.xticks(rotation=45)
plt.axhline(
    y=4, color='orange', linestyle='--', linewidth=2, label='Goal: 4'
)
plt.legend()
plt.show()

**What is the 4 week rolling S/I average for each practitioner?**

In [ ]:
df_clean['rolling_4-week_S/I_ratio'] = pd.to_numeric(
    df_clean['rolling_4-week_S/I_ratio'], errors='coerce'
)

practitioner_trend = df_clean.groupby('clinician_id')['rolling_4-week_S/I_ratio'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))

practitioner_trend.plot(kind ='barh', ax=ax, color='#4C72B0')
plt.title('Practitioner Average 4 Week S/I Ratio Trend')
plt.xlabel('S/I Ratio')
plt.ylabel('Practitioner')
plt.xticks(rotation=45)
plt.axvline(
    x=4, color='orange', linestyle='--', linewidth=2, label='Goal: 4'
)
plt.legend()
plt.show()

In [ ]:
df_clean['rolling_4-week_S/I_ratio'].dtype

**What does the 4 week trend for each clinician look like?**

In [ ]:
# Build chronological date to use repeatedly, utilize real date to sort and index iso_week.
week_order = df_clean.groupby('iso_week')['real_date'].min().sort_values().index

fig, ax = plt.subplots(figsize=(12, 6))

for clinician in practitioner_trend.index:
    #build our group
    group = df_clean[df_clean['clinician_id'] == clinician]
    #calculate weekly rolling average, should start after 4 weeks of data
    weekly = group.groupby('iso_week')['rolling_4-week_S/I_ratio'].mean()
    #bring in chronological date variable to ensure correct order
    weekly = weekly.reindex(week_order)
    weekly.plot(ax=ax, marker='o', label=clinician, alpha=0.7)

ax.set_xlabel('Week')
ax.set_ylabel('Rolling 4-Week S/I Ratio')
ax.set_title('S/I Ratio Trend by Practitioner Over Time')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=45)
plt.axhline(
    y=4, color='black', linestyle='--', linewidth=2, label='Goal: 4'
)
plt.tight_layout()
plt.show()


This graphic appears chaotic and noisy, which is precisely why giving the business owner the ability to sort by practioners (as requested) was a top priority in the visualizations.

In [ ]:
tenure_si_summary = tenure_si_summary.merge(return_rate_by_clinician.reset_index() if isinstance(return_rate_by_clinician, pd.Series) else return_rate_by_clinician, on='clinician_id', how='left')

print(tenure_si_summary)

**Pearson coefficient measures the linear relationship between continuous variables. Spearman coefficient measures the strength and direction of a monotonic relationship between two variables — meaning, does one variable consistently tend to increase (or decrease) as the other increases, regardless of whether that relationship is a straight line.**

In [ ]:
corr_return, p_return = stats.pearsonr(tenure_si_summary['second_visit_return_rate'], tenure_si_summary['avg_si_ratio'])
print(f"Return rate vs S/I ratio — Pearson r = {corr_return:.3f}, p = {p_return:.3f}")

**Looking at the simple second visit return rate vs S/I ratio, there is almost no correlation and the results are not statistically significant.**

**Looking at clinician I, I want to see if low patient count correlates to extreme S/I ratio**

In [ ]:
#Looking at clinician I, I want to see if low patient count correlates to extreme S/I ratio
print(tenure_si_summary)

**There doesn't necessarily appear to be a correlation, but let's dig deeper.**

In [ ]:


# Correlation: caseload size vs S/I ratio
corr_caseload, p_caseload = stats.pearsonr(
    tenure_si_summary['patients_per_clinician'],
    tenure_si_summary['avg_si_ratio']
)
print(f"Caseload vs S/I ratio — Pearson r = {corr_caseload:.3f}, p = {p_caseload:.3f}, n = {len(tenure_si_summary)}")

spear_caseload, spear_p_caseload = stats.spearmanr(
    tenure_si_summary['patients_per_clinician'],
    tenure_si_summary['avg_si_ratio']
)
print(f"Caseload vs S/I ratio — Spearman r = {spear_caseload:.3f}, p = {spear_p_caseload:.3f}")

**Looking at Pearson coefficient, there appears to be a weak, negative correlation between caseload and S/I ratio. Additionally, The Spearman coefficient shows an even weaker correlation between the two.**

**Z-Score in statistics measures how many standard deviations a data point lies away from the mean of a distribution. It standardizes values across different distributions, enabling meaningful comparisons even when datasets have different means and standard deviations. It is widely used in hypothesis testing, outlier detection and normalizing data for machine learning models.**

https://www.geeksforgeeks.org/data-science/z-score-in-statistics/

Since clinician I appears to have an extreme S/I ratio, let's run this measure to see about the possibility of exclusion from the dataset.

In [ ]:
# Z-score calculation: flag clinicians whose avg_si_ratio is unusually far from the group mean
mean_ratio = tenure_si_summary['avg_si_ratio'].mean()
std_ratio = tenure_si_summary['avg_si_ratio'].std()

tenure_si_summary['z_score'] = (tenure_si_summary['avg_si_ratio'] - mean_ratio) / std_ratio

print(tenure_si_summary[['clinician_id', 'patients_per_clinician', 'avg_si_ratio', 'z_score']].sort_values('z_score', ascending=False))

In [ ]:
Z_THRESHOLD= 2

#determine outlier clinicians
outlier_summary = tenure_si_summary[tenure_si_summary['z_score'].abs() > Z_THRESHOLD]
reliable_tenure_summary = tenure_si_summary[tenure_si_summary['z_score'].abs() <= Z_THRESHOLD]

print(f'Number of clinicians excluded: {len(outlier_summary)}')
print(f'The following clinicians fall outside the Z_THRESHOLD: {outlier_summary['clinician_id']}')
print(f'Number of reliable clinicians: {len(reliable_tenure_summary)}')


In [ ]:


# Full dataset (before outlier removal), for comparison
full_clean = tenure_si_summary.dropna(subset=['tenure_years'])
corr_full, p_full = stats.pearsonr(full_clean['tenure_years'], full_clean['avg_si_ratio'])
print("=== Full dataset ===")
print(f"n = {len(full_clean)}")
print(f"Pearson r = {corr_full:.3f}, p = {p_full:.3f}")
print()

# Filtered dataset (outlier removed, matches your established methodology pattern)
reliable_clean = reliable_tenure_summary.dropna(subset=['tenure_years'])
corr_filtered, p_filtered = stats.pearsonr(reliable_clean['tenure_years'], reliable_clean['avg_si_ratio'])
print("=== Filtered dataset (outlier removed) ===")
print(f"n = {len(reliable_clean)}")
print(f"Pearson r = {corr_filtered:.3f}, p = {p_filtered:.3f}")
print()

# Spearman as a robustness check on the filtered dataset
spear_corr, spear_p = stats.spearmanr(reliable_clean['tenure_years'], reliable_clean['avg_si_ratio'])
print("=== Spearman (filtered) ===")
print(f"r = {spear_corr:.3f}, p = {spear_p:.3f}")

**The z-score outlier exclusion was useful to run to cofirm my suspicions, but it did not alter the correlation results, as the flagged outlier (Clinician I) was already excluded due to missing tenure data.**

**The Pearson score for the dataset shows a strong, positive, statistically significant correlation between tenure and S/I ratio. The Spearman score also shows a strong and significant correlation.**

In [ ]:
!git status

In [ ]:

!git add 'SI_Ratio_Analysis.ipynb'
!git commit -m 'Notebook cleaned up. Analysis complete?'
!git push